# Window Average Model

This tutorial explains how to use the **WindowAverage** forecasting model. WindowAverage is a simple baseline model that serves as a reference point for more advanced models like ARIMA, ETS, and TBATS.

In [1]:
import sys
import os

# Add parent directory to path so we can import the library modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import jax
import jax.numpy as jnp
import numpy as np

from window_average import WindowAverage
from conformal_intervals import ConformalIntervals

## Model Overview

WindowAverage forecasts future values by computing the mean of the most recent *k* observations (a fixed-size sliding window).

## Mathematical Overview

For a window size k:

$\hat{y}_{t+1} = \frac{1}{k} \sum_{i=t-k+1}^{t} y_i$

## When to Use

**Use when:**
* You need a simple baseline forecast for benchmarking
* Short-horizon predictions on stationary series
* Quick sanity checks before fitting more complex models

**Avoid when:**
* Strong trend or seasonality is present
* Long-horizon forecasts are needed
* Structural breaks exist in the data

## Brief Implementation Details
* Stores the training series during fit
* Computes window mean at prediction time
* Stateful forecasting (fit + predict)
* Stateless forecasting (forecast)
* Conformal Intervals for uncertainty estimation

## API Contract

### Constructor

```python
WindowAverage(
    window_size: int,
    conformal_params: Optional[ConformalIntervals] = None
)
```

### What are conformal parameters?

```python
ConformalIntervals(
    n_windows: int,
    h: int,
    method: str
)
```

- `n_windows`: Number of rolling windows
- `h`: Forecast horizon for conformal calibration
- `method`: Conformal method to create intervals. Default is `"conformal_distribution"` → symmetrical intervals

Conformal intervals are uncertainty estimations around point forecasts based on historical forecast errors.

### Methods
1. `fit(y)`: Stores training series
2. `predict(h, level)`: Forecast using fitted model
3. `forecast(y, h, level, fitted=False)`: Stateless forecasting

### Fit and Predict

#### Fit Method

```python
fit(y: ArrayLike) -> Self
```
`y`: historical time series

Fit stores the series and prepares the model for forecasting. No optimization of parameters needed at this step.

#### Predict Method

```python
predict(h: int, level: Optional[List[int]] = None) -> Dict[str, ArrayLike]
```
- `h`: forecast horizon
- `level`: Confidence intervals for conformal prediction intervals

Generates future forecasts using the fitted model. Produces point forecasts and conformal prediction intervals if asked.

#### Example: Fit + Predict

In [2]:
# Basic fit and predict example
y = jnp.asarray([1., 3., 2., 5., 4., 6.])

m = WindowAverage(
    window_size=2,
    conformal_params=ConformalIntervals(
        n_windows=2,
        h=1,
        method="conformal_distribution"
    )
)

m.fit(y)
out = m.predict(h=2, level=[50, 80, 95])

print("Point forecast:", out["mean"])
print("Lower 80%:", out["lo-80"])
print("Upper 80%:", out["hi-80"])
print("Forecast shape:", out["mean"].shape)

Point forecast: [5. 5.]
Lower 80%: [3.8 3.8]
Upper 80%: [6.2 6.2]
Forecast shape: (2,)


### Forecast (Stateless Forecasting)

Stateless forecasting performs prediction without reading from or changing any internal model state.

Stateless forecasting is functionally the same as calling fit then predict, but does not persist any model data.

```python
forecast(
    y: ArrayLike,
    h: int,
    level: Optional[List[int]] = None,
    fitted: bool = False
)
```
- `y`: Time series
- `h`: Forecast horizon
- `level`: Confidence intervals for conformal prediction intervals
- `fitted`: Should we reuse previously fitted state

#### Example: Stateless Forecast with Conformal Intervals

In [3]:
# Stateless forecast with conformal intervals
# Enough samples for conformity scoring: (n - 1) // h >= 2
y = jnp.asarray([1., 2., 3., 6., 9., 9., 8., 7.])

# Initialize conformal intervals
cfg = ConformalIntervals(
    n_windows=3,
    h=1,
    method="conformal_distribution"
)

m = WindowAverage(
    window_size=3,
    conformal_params=cfg
)

out = m.forecast(
    y=y,
    h=4,
    level=[90],
    fitted=False
)

print("Forecast mean:", out["mean"])
print("Lower 90%:", out["lo-90"])
print("Upper 90%:", out["hi-90"])

# Verify intervals are consistent
assert "mean" in out and out["mean"].shape == (4,)
assert "lo-90" in out and "hi-90" in out
assert jnp.all(out["lo-90"] <= out["mean"])
assert jnp.all(out["mean"] <= out["hi-90"])

print("\nAll assertions passed! Intervals are consistent.")

Forecast mean: [8. 8. 8. 8.]
Lower 90%: [6.58333302 6.58333302 6.58333302 6.58333302]
Upper 90%: [9.41666698 9.41666698 9.41666698 9.41666698]

All assertions passed! Intervals are consistent.


## Using Fit + Predict vs Forecast

Use **fit + predict** when you want to fit once and then generate forecasts interactively — ideal for exploration and debugging.

Use **forecast** when you need a single stateless call for repeated or batch evaluation — cross-validation, distributed pipelines, or reproducible experiments.

## Edge Cases and Limitations

It's best to use WindowAverage as a baseline or benchmarking tool or for simple, stationary/smooth series.

**Edge Cases:**
1. Short time series: If length of series < window size, forecast returns NaN values
2. Conformal intervals require `2h + 1` observations per rolling window
3. Unsupported `fitted = True` in forecast

**Limitations of model:**
1. Cannot model trend nor seasonality
2. Constant multi-step forecasts
3. Sensitivity to window size: Small windows can result in noisier forecasts, but larger windows can also minimize any important local patterns.
4. Prediction intervals based on past errors: Conformal intervals assume that forecast errors can represent future errors but any sudden changes in our series can ruin representativeness of intervals.